# 03 — Customer Segmentation and Association Rules

K-Means segmentation (k=4, chosen by silhouette + Davies-Bouldin agreement, overriding the statistically higher-scoring but business-thin k=2 -- see the methodology reasoning in the markdown cells below) and FP-Growth association rule mining.

In [2]:

from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb pandas pyarrow scikit-learn xgboost lightgbm mlxtend shap prophet google-genai

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROCESSED_DIR / "features"
MODELS_DIR = PROCESSED_DIR / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
WAREHOUSE_PATH = PROJECT_ROOT / "warehouse.duckdb"

for d in [PROCESSED_DIR, FEATURES_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/Ecommerce-AI-Business-Intelligence


In [3]:

import pandas as pd
customer_features = pd.read_parquet(FEATURES_DIR / "customer_features.parquet")
customer_features = customer_features.dropna(subset=["recency_days", "frequency", "monetary"])
print(f"Clustering on {len(customer_features):,} customers")


Clustering on 95,420 customers


## K selection: silhouette + Davies-Bouldin, with a minimum-cluster-size guard

`k=2` typically scores highest on silhouette but only rediscovers a trivial 'bought more than once or not' split. `k` values whose smallest cluster falls below `MIN_CLUSTER_FRACTION` are excluded from automatic selection; `FORCE_K` lets you override after reviewing the table.

In [4]:

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

RFM_COLS = ["recency_days", "frequency", "monetary"]
K_RANGE = range(2, 9)
RANDOM_STATE = 42
SILHOUETTE_SAMPLE_SIZE = 5000
MIN_CLUSTER_FRACTION = 0.03
FORCE_K = 4  # set from prior analysis: k=4 is the only k>=3 where silhouette and Davies-Bouldin agree

scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_features[RFM_COLS])

scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels, sample_size=SILHOUETTE_SAMPLE_SIZE, random_state=RANDOM_STATE)
    db = davies_bouldin_score(X_scaled, labels)
    min_cluster_pct = pd.Series(labels).value_counts().min() / len(labels)
    scores.append({"k": k, "silhouette": sil, "davies_bouldin": db, "min_cluster_pct": min_cluster_pct})
    print(f"k={k}  silhouette={sil:.4f}  davies_bouldin={db:.4f}  smallest_cluster={min_cluster_pct*100:.1f}%")

scores_df = pd.DataFrame(scores)
best_k = FORCE_K if FORCE_K else int(
    scores_df[scores_df["min_cluster_pct"] >= MIN_CLUSTER_FRACTION].sort_values("silhouette", ascending=False).iloc[0]["k"])
print(f"\nUsing k={best_k}")


k=2  silhouette=0.7377  davies_bouldin=0.5561  smallest_cluster=3.1%
k=3  silhouette=0.4546  davies_bouldin=0.7609  smallest_cluster=3.1%
k=4  silhouette=0.4871  davies_bouldin=0.6835  smallest_cluster=2.6%
k=5  silhouette=0.4133  davies_bouldin=0.7552  smallest_cluster=2.5%
k=6  silhouette=0.4320  davies_bouldin=0.7039  smallest_cluster=0.6%
k=7  silhouette=0.4349  davies_bouldin=0.7268  smallest_cluster=0.3%
k=8  silhouette=0.4435  davies_bouldin=0.7510  smallest_cluster=0.3%

Using k=4


In [5]:

final_model = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
customer_features["segment_id"] = final_model.fit_predict(X_scaled)

profile = customer_features.groupby("segment_id").agg(
    n_customers=("customer_unique_id", "count"),
    avg_recency_days=("recency_days", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
).reset_index().sort_values("avg_monetary", ascending=False)
profile["pct_of_customers"] = (profile["n_customers"] / profile["n_customers"].sum() * 100).round(1)

SEGMENT_LABELS = {0: "Lapsed one-time buyers", 1: "Recent one-time buyers",
                  2: "Loyal repeat customers", 3: "High-value one-time buyers"}
profile["segment_label"] = profile["segment_id"].map(SEGMENT_LABELS)
# NOTE: verify this mapping against YOUR run's actual profile stats below --
# label assignment depends on which cluster index gets which characteristics,
# which can vary by run. Adjust SEGMENT_LABELS if needed before trusting the labels.
display(profile)

customer_features[["customer_unique_id", "segment_id"] + RFM_COLS].to_parquet(
    MODELS_DIR / "customer_segments.parquet", index=False)
profile.to_parquet(MODELS_DIR / "segment_profile.parquet", index=False)


,segment_id,n_customers,avg_recency_days,avg_frequency,avg_monetary,pct_of_customers,segment_label
3,3,2466,246.056772,1.013788,1173.024294,2.6,High-value one-time buyers
2,2,2881,227.086428,2.114891,289.619729,3.0,Loyal repeat customers
1,1,51817,134.475790,1.000000,134.709528,54.3,Recent one-time buyers
0,0,38256,394.740302,1.000000,134.330934,40.1,Lapsed one-time buyers


## Association rule mining (FP-Growth)

Olist orders are predominantly single-category -- expect a thin or null result. The retry logic below lowers the support threshold automatically if nothing is found, down to a floor, before concluding there's genuinely no cross-category signal (a legitimate, citable finding, not a failure).

In [6]:

from mlxtend.frequent_patterns import association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder

order_baskets = pd.read_parquet(FEATURES_DIR / "order_baskets.parquet")
transactions = order_baskets.groupby("order_id")["category_name_english"].apply(list).tolist()

basket_sizes = pd.Series([len(t) for t in transactions])
print("Basket size distribution:\n", basket_sizes.value_counts().sort_index())
print(f"{(basket_sizes > 1).sum()} of {len(transactions)} baskets ({100*(basket_sizes>1).mean():.1f}%) are multi-category")

encoder = TransactionEncoder()
encoded_df = pd.DataFrame(encoder.fit_transform(transactions), columns=encoder.columns_)

MIN_SUPPORT, MIN_SUPPORT_FLOOR, MIN_LIFT, MIN_LIFT_FLOOR = 0.005, 0.00005, 1.2, 1.0
support = MIN_SUPPORT
while True:
    frequent_itemsets = fpgrowth(encoded_df, min_support=support, use_colnames=True)
    has_multi = not frequent_itemsets.empty and (frequent_itemsets["itemsets"].apply(len) >= 2).any()
    if has_multi or support <= MIN_SUPPORT_FLOOR:
        break
    support = max(support * 0.5, MIN_SUPPORT_FLOOR)

print(f"\nFrequent itemsets at support={support:.5f}: {len(frequent_itemsets)} "
      f"({(frequent_itemsets['itemsets'].apply(len) >= 2).sum() if not frequent_itemsets.empty else 0} multi-category)")

if not frequent_itemsets.empty and (frequent_itemsets["itemsets"].apply(len) >= 2).any():
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=MIN_LIFT)
    print(f"Rules with lift > {MIN_LIFT}: {len(rules)}")
    display(rules.sort_values("lift", ascending=False).head(10))
else:
    print("No multi-category itemsets found -- consistent with Olist's overwhelmingly single-category purchasing "
          "behavior. This is a legitimate finding, not a bug (see methodology notes).")


Basket size distribution:
 1    96530
2      711
3       15
Name: count, dtype: int64
726 of 97256 baskets (0.7%) are multi-category


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


Frequent itemsets at support=0.00063: 57 (1 multi-category)
Rules with lift > 1.2: 0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag